# Daisyworld Tutorial

<div align="center">
    <img src="../docs/assets/daisyworld_defaults.png">
</div>


## Summary

Daisyworld is a simple, 0-dimensional model of planetary climate regulation via uncoordinated control. Any particular instance is defined by three state values that change over time: the ground covered by two different species of daisies, $\alpha_d$ and $\alpha_l$, and stellar luminosity $L$, a unitless forcing factor that determines how much incident stellar radiation reaches the planet.

To run the model, we need three differential equations to describe how the three state values change over time. But one of these, describing how $L$ changes, is just a constant value indicating a simple linear ramp as $L$ increases (typically until reaching some maximum $L_{max}$). The other two are actually the same equation that take different values, one for each of the two daisy species in a default Daisyworld model. Five other equations define how we get the inputs to the three differential equations from Daisyworld parameters.

In this tutorial we will implement each of these equations in turn to build a Daisyworld from scratch. 

## Equations reference

_These are here to refer to while implementing Daisyworld. For more details, check out [the paper](https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1600-0889.1983.tb00031.x) or [Lovelock's archived version](https://www.jameslovelock.org/biological-homeostasis-of-the-global-environment-the-parable-of-daisyworld/)_.

### Rate of change in stellar luminosity, the forcing factor

If $L < L_{max}$:

$$
\frac{dL}{dt} = \Delta L
$$
<div align="right">
    (0a)
</div>


Otherwise:

$$
\frac{dL}{dt} = 0
$$
<div align="right">
    (0b)
</div>


Where $L$ is the stellar luminosity and $\Delta L$ is the amount of change per unit time. To get the value of $L$ at the next time step $t_{next}$, increment the current value $L_{t_{now}}$ by the rate of change proportional to the time step size $\Delta t$, that is $L_{t_{next}} = L_{t_{now}} + \Delta t \frac{dL}{dt}$. In this tutorial we'll use a value of $\delta T$ for both equation 0 and 1, but note that in the implementation `dw.simple_dw` in this repository there is no explicit $\Delta t$ used for changes in L, just a simple linear ramp based on the starting and end values of L. 

### Rate of change in daisy population (ground cover)

$$
\frac{d\alpha_k}{dt} = \alpha_k (x\beta_k - \gamma)
$$
<div align="right">
    (1)
</div>

Where $\alpha_k$ is the ground coverage of daisy species $k$, $\beta_k$ is the temperature dependent growth rate for daisy speices $k$ (equation 3), and $\gamma$ is a constant proportional death rate shared by both daisy species, _e.g._ 0.05 for 5% loss per unit time. $$x$ is the free ground available for daisies to grow on that is both capable of supporting daisies and not occupied by either daisy species(equation 2).

### Ground available for daisy growth

$$
x = p - \alpha_{d} - \alpha_{l}
$$
<div align="right">
    (2a)
</div>

Where $p$ is the proportion of land that is arable, or capable of supporting daisies. $\alpha_d$ and $\alpha_l$ are the proportions of ground covered by dark and light species, respectively. The original Daisyworld had only two daisy species, black and white (actually both different shades of gray), but in principle we could have many species of daisy each with a different albedo. The general equation for calculating $x$ is to subtract the sum of ground coverage for all $K$ daisy species from the proportion of ground that is arable, $p$.

$$
x = p - \sum_k^K{\alpha_k}
$$
<div align="right">
    (2b)
</div>

### Temperature-dependent growth rate

When $g(T_o-T_k)^2 \leq 1.0$:

$$
\beta_k = 1 - g(T_{o} - T_k)^2
$$
<div align="right">
    (3a)
</div>

and otherwise 
$$
\beta_k = 0
$$
<div align="right">
    (3b)
</div>

Where $\beta_k$ is the temperature-dependent growth rate for daisy species $k$, $T_o$ is the optimal temperature in Kelvin (295.5 K), $T-k$ is the local temperature experienced by daisy species $k$, and $g$ is a constant that determines the range of temperatures that allow daisies to grow. We'll use a default value of $g$ that allows daisy growth from 278 K to 313 K is $g = 1/17.5^2 \approx 0.03625$.

### Effective temperature 

$$
T_e = \left (\frac{SL(1-A)}{\sigma} \right )^{1/4}
$$
<div align="right">
    (4)
</div>

Where $T_e$ is the effective temperature of Daisyworld, $S$ is the strength of stellar radiation in Watts per square meter (we use a value of 1000 $\frac{W}{m^2}$), $L$ is unitless luminosity which we use to increase stellar radiation over time, $\sigma$ is the [Stefan-Boltzmann constant](https://en.wikipedia.org/wiki/Stefan%E2%80%93Boltzmann_law) with units of Watts divided by the product of meters squared and degrees Kelvin raised to the power of 4 , $\approx 5.67\times10^-8 \frac{W}{m^2K^4}$

The effective temperature $T_e$ ensures the energy balance of Daisyworld, as the amount of energy due to incident stellar radiation must be re-radiated by Daisyworld.

### Planetary albedo

$$
A = \alpha_g A_g + \alpha_d A_d + \alpha_l A_l
$$
<div align="right">
    (5a)
</div>

The planetary albedo $A$ determines how much stellar radiation is reflected back into space by Daisyworld. $A_g$ and $\alpha_g$ are the bare ground albedo and proportion of ground which is bare, respectively. $\alpha_d$ is the proportion of ground covered by dark and $\alpha_l$ the proportion covered by light daisies, while $A_d$ and $A_l$ are the respective albedos of each daisy species. The general equation, which covers situations with greater than or fewer than two daisy species is the sum of all daisy albedos weighted by the ground covered by each. 

$$
A = \alpha_g A_g + \sum_k^K{\alpha_k A_k}
$$
<div align="right">
    (5b)
</div>

### Local temperature (experienced by each daisy species)

$$
T_k = \left( q(A - A_k) + T_e^4 \right )^{1/4}
$$
<div align="right">
    (6)
</div>

The local temperature experience by each daisy species is $T_k$, and calculated using equation 6 above. The only new variable to consider here is $q$, which has units of $\frac{1}{K^4}$ and determines the temperature difference between regions of Daisyworld with different albedos. As noted in Watson and Lovelock's 1983 paper, for $q=0$, there is perfect conduction and all of Daisyworld experiences the same temperature (and consequently, the same growth rate for all daisy species). A value greater than $\frac{SL}{\sigma}$ would mean that heat flows from the higher albedo (cooler) regions to the darker (hotter) regions: not a realistic scenario. We'll use $q = 0.2 \frac{SL}{\sigma}$ as in the 1983 paper. 

<!-- ### Variables (table)-->